In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
import time
import gc
import csv

print("--- STARTING KAGGLE BATCH EXECUTION: DEEP CONV-GRU ---")

# ==========================================
# PART 1: DATA PIPELINE
# ==========================================
print("\n[1/3] Initializing Memory-Safe Data Pipeline...")

DATASET_PATH = "/kaggle/input/datasets/yashaswi15/aerosense-training-data/Delhi_NCR_Master_DataCube_Scaled.npy"

master_data = np.load(DATASET_PATH, mmap_mode='r')
total_days = master_data.shape[0]
train_split = int(total_days * 0.8) 
test_split = total_days - train_split 

class SpatiotemporalGenerator(Sequence):
    def __init__(self, data_cube, start_idx, end_idx, batch_size=16):
        self.data_cube = data_cube
        self.start_idx = start_idx
        self.end_idx = end_idx - 8 
        self.batch_size = batch_size
        self.indices = np.arange(self.start_idx, self.end_idx)
        
    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        X_batch, Y_batch = [], []
        
        for i in batch_indices:
            window = self.data_cube[i : i+7]
            X_batch.append(window)
            Y_batch.append(self.data_cube[i+7])
            
        X_batch = np.nan_to_num(np.array(X_batch), nan=0.0).astype('float32')
        Y_batch = np.nan_to_num(np.array(Y_batch), nan=0.0).astype('float32')
        return X_batch, Y_batch

train_gen = SpatiotemporalGenerator(master_data, 0, train_split, batch_size=16)
val_gen = SpatiotemporalGenerator(master_data, train_split, total_days, batch_size=16)

# ==========================================
# PART 2: BUILDING CUSTOM CONV-GRU LAYER 
# ==========================================
print("\n[2/3] Building Custom ConvGRU Architecture (Object-Oriented Layer)...")

class SpatiotemporalConvGRU(layers.Layer):
    def __init__(self, filters, **kwargs):
        super(SpatiotemporalConvGRU, self).__init__(**kwargs)
        self.filters = filters
        self.conv_z = layers.Conv2D(filters, 3, padding='same', activation='sigmoid', name='Update_Gate')
        self.conv_r = layers.Conv2D(filters, 3, padding='same', activation='sigmoid', name='Reset_Gate')
        self.conv_h = layers.Conv2D(filters, 3, padding='same', activation='tanh', name='Candidate_H')

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        h = tf.zeros((batch_size, 141, 231, self.filters))

        for t in range(7):
            x_t = inputs[:, t, :, :, :] 
            concat_xh = tf.concat([x_t, h], axis=-1)
            z = self.conv_z(concat_xh)
            r = self.conv_r(concat_xh)
            concat_x_rh = tf.concat([x_t, r * h], axis=-1)
            h_tilde = self.conv_h(concat_x_rh)
            h = (1.0 - z) * h + z * h_tilde
            
        return h

def build_conv_gru_model(input_shape=(7, 141, 231, 6), filters=32):
    inputs = layers.Input(shape=input_shape)
    h = SpatiotemporalConvGRU(filters=filters)(inputs)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(h)
    outputs = layers.Conv2D(6, 1, activation='sigmoid')(x)
    return Model(inputs=[inputs], outputs=[outputs], name="Deep_ConvGRU_Custom")

model = build_conv_gru_model()


# ==========================================
# PART 3: ADVANCED CUSTOM TRAINING LOOP (WITH NATIVE CSV LOGGER)
# ==========================================
MODEL_NAME = "Deep_ConvGRU" 
MODEL_SAVE_PATH = f"/kaggle/working/Delhi_NCR_{MODEL_NAME}_Best.keras"
CSV_LOG_PATH = f"/kaggle/working/{MODEL_NAME}_Training_Log.csv"

with open(CSV_LOG_PATH, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss", "val_mape"])

print(f"\n[3/3] Initiating Custom Training Loop for {MODEL_NAME}...")
print(f" -> Metrics will be safely logged to: {CSV_LOG_PATH}")

initial_learning_rate = 0.001
decay_steps = 50 * len(train_gen)
lr_schedule = CosineDecay(initial_learning_rate, decay_steps)
optimizer = Adam(learning_rate=lr_schedule, clipnorm=1.0) # Crucial for custom RNNs

def calculate_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.abs(y_true - y_pred))

@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        predictions = model(x_batch, training=True)
        loss = calculate_loss(y_batch, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

@tf.function
def val_step(x_batch, y_batch):
    predictions = model(x_batch, training=False)
    val_loss = calculate_loss(y_batch, predictions)
    
    y_true_f = tf.cast(y_batch, tf.float32)
    pred_f = tf.cast(predictions, tf.float32)
    mape = tf.reduce_mean(tf.abs((y_true_f - pred_f) / (y_true_f + 1e-10))) * 100.0
    
    return val_loss, mape

EPOCHS = 50
best_val_loss = float('inf')
patience = 7
patience_counter = 0

print(f"\n--- Starting 50-Epoch Backpropagation ---")

for epoch in range(EPOCHS):
    start_time = time.time()
    epoch_loss_avg = tf.keras.metrics.Mean()
    epoch_val_loss_avg = tf.keras.metrics.Mean()
    epoch_val_mape_avg = tf.keras.metrics.Mean() 
    
    for step in range(len(train_gen)):
        x_batch, y_batch = train_gen[step]
        loss_val = train_step(x_batch, y_batch)
        epoch_loss_avg.update_state(loss_val)
        
    for step in range(len(val_gen)):
        x_val, y_val = val_gen[step]
        v_loss, v_mape = val_step(x_val, y_val)
        epoch_val_loss_avg.update_state(v_loss)
        epoch_val_mape_avg.update_state(v_mape)
        
    train_loss = epoch_loss_avg.result().numpy()
    val_loss = epoch_val_loss_avg.result().numpy()
    val_mape = epoch_val_mape_avg.result().numpy()
    
    current_lr = optimizer.learning_rate(optimizer.iterations).numpy() if callable(optimizer.learning_rate) else optimizer.learning_rate.numpy()
        
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Time: {time.time() - start_time:.1f}s | LR: {current_lr:.5f} | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f} | Val MAPE: {val_mape:.2f}%")
    
    # APPEND METRICS TO NATIVE CSV FILE
    with open(CSV_LOG_PATH, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch + 1, train_loss, val_loss, val_mape])
    
    if val_loss < best_val_loss:
        print(f"  -> Val Loss improved from {best_val_loss:.5f} to {val_loss:.5f}. Saving weights!")
        best_val_loss = val_loss
        model.save(MODEL_SAVE_PATH)
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\n[!] Early Stopping Triggered.")
            break
            
    gc.collect()
    tf.keras.backend.clear_session()

print(f"\nTraining Complete! Logs successfully written to {CSV_LOG_PATH}")

# ==========================================
# PART 4: 20% TEST SET EVALUATION
# ==========================================
print("\n[4/4] Evaluating Best ConvGRU Weights on 20% Test Set...")

# Load the absolute best weights before evaluating
model.load_weights(MODEL_SAVE_PATH)

CO_MIN = 0.0195770263671875  
CO_MAX = 0.11248779296875 

true_co_list = []
pred_co_list = []

for i in range(train_split, total_days - 8):
    window = master_data[i : i+7]
    X_input = window.reshape((1, 7, 141, 231, 6))
    X_input = np.nan_to_num(X_input, nan=0.0).astype('float32')
    Y_real = master_data[i+7]
    
    Y_pred = model.predict(X_input, verbose=0)[0]
    
    real_co = Y_real[:, :, 2] * (CO_MAX - CO_MIN) + CO_MIN
    pred_co = Y_pred[:, :, 2] * (CO_MAX - CO_MIN) + CO_MIN
    
    true_co_list.append(np.mean(real_co))
    pred_co_list.append(np.mean(pred_co))
    
    gc.collect()

y_true = np.array(true_co_list)
y_pred = np.array(pred_co_list)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
threshold = np.mean(y_true)
accuracy_pct = accuracy_score((y_true > threshold).astype(int), (y_pred > threshold).astype(int)) * 100

print("\n" + "="*50)
print("       === DEEP CONV-GRU 20% TEST SET EVALUATION ===")
print("="*50)
print(f"Mean Absolute Error (MAE)         : {mae:.6f} mol/m²")
print(f"Root Mean Squared Error (RMSE)    : {rmse:.6f} mol/m²")
print(f"High-Pollution Detection Accuracy : {accuracy_pct:.2f}%")
print("="*50)

2026-04-23 16:59:29.528888: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776963569.725584      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776963569.779324      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776963570.207555      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776963570.207598      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776963570.207601      22 computation_placer.cc:177] computation placer alr

--- STARTING KAGGLE BATCH EXECUTION: DEEP CONV-GRU ---

[1/3] Initializing Memory-Safe Data Pipeline...

[2/3] Building Custom ConvGRU Architecture (Object-Oriented Layer)...


I0000 00:00:1776963598.330071      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



[3/3] Initiating Custom Training Loop for Deep_ConvGRU...
 -> Metrics will be safely logged to: /kaggle/working/Deep_ConvGRU_Training_Log.csv

--- Starting 50-Epoch Backpropagation ---


I0000 00:00:1776963606.502370      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


Epoch 01/50 | Time: 140.7s | LR: 0.00100 | Train Loss: 0.07424 | Val Loss: 0.04954 | Val MAPE: 1874768384.00%
  -> Val Loss improved from inf to 0.04954. Saving weights!
Epoch 02/50 | Time: 120.9s | LR: 0.00100 | Train Loss: 0.04841 | Val Loss: 0.04714 | Val MAPE: 1663757568.00%
  -> Val Loss improved from 0.04954 to 0.04714. Saving weights!
Epoch 03/50 | Time: 120.6s | LR: 0.00099 | Train Loss: 0.04642 | Val Loss: 0.04498 | Val MAPE: 1357770112.00%
  -> Val Loss improved from 0.04714 to 0.04498. Saving weights!
Epoch 04/50 | Time: 119.6s | LR: 0.00098 | Train Loss: 0.04473 | Val Loss: 0.04314 | Val MAPE: 1190174080.00%
  -> Val Loss improved from 0.04498 to 0.04314. Saving weights!
Epoch 05/50 | Time: 118.6s | LR: 0.00098 | Train Loss: 0.04304 | Val Loss: 0.04117 | Val MAPE: 965039104.00%
  -> Val Loss improved from 0.04314 to 0.04117. Saving weights!
Epoch 06/50 | Time: 118.4s | LR: 0.00096 | Train Loss: 0.04169 | Val Loss: 0.04036 | Val MAPE: 746739200.00%
  -> Val Loss improved fro

I0000 00:00:1776969738.100175      68 service.cc:152] XLA service 0x7dad04001380 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776969738.100284      68 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1776969739.028684      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



       === DEEP CONV-GRU 20% TEST SET EVALUATION ===
Mean Absolute Error (MAE)         : 0.001347 mol/m²
Root Mean Squared Error (RMSE)    : 0.001944 mol/m²
High-Pollution Detection Accuracy : 89.25%
